In [1]:
import os, sys

print("Aantal paden in sys.path:", len(sys.path))
# Bepaal de parent-werkmap
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

#Loop door alle submappen en voeg ze toe aan sys.path
for dirpath, dirnames, filenames in os.walk(parent_dir):
    if dirpath not in sys.path:
        sys.path.insert(0, dirpath)

print("Aantal paden in sys.path:", len(sys.path))

Aantal paden in sys.path: 9
Aantal paden in sys.path: 320


In [2]:
from Libraries.inference_training import Configuration, ImageDataset
from Libraries.inference_training import initCudaEnvironment, createTransforms
from Libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from Libraries.inference_training import trainModel, saveModel, loadModel
import random
import sys
from pathlib import Path
print(str(Path().resolve().parents[1]))
from paths import *


C:\Users\Tomkr\Jaar2BlokD\Tygron


In [3]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

In [4]:
def createModel(trainDirectory: str, testDirectory: str, modelName: str, epochs: int, labels: list[str], augment_data: bool, save_path: str, save_interval=0, maskdata=None , model_description="default"):
    """
    Create, configure, train, and export an instance segmentation model using a configurable setup.

    This function prepares the dataset, configures the training pipeline, trains the model for the
    specified number of epochs, and exports the resulting model to both PyTorch and ONNX formats.

    Parameters:
    -----------
    trainDirectory : str
        Path to the training dataset folder.
    testDirectory : str
        Path to the testing dataset folder.
    modelName : str
        Name assigned to the trained model and saved output files.
    epochs : int
        Number of training epochs.
    labels : list[str]
        List of label names used for segmentation. For example, ['parking_spaces'].
    augment_data : bool
        Whether to apply data augmentation during training.
    save_path : str
        Directory path where the model will be saved.
    save_interval : int, optional
        Interval (in epochs) at which to save model checkpoints. Default is 0 (disabled).
    maskdata : list[float], optional
        List containing three float values:
        [scoreThreshold, maskThreshold, strideFraction] used for ONNX metadata.
        Defaults to [0.2, 0.3, 0.5] if not provided.
    model_description : str, optional
        Description to embed into the ONNX metadata. Default is "default".

    Returns:
    --------
    model : torch.nn.Module
        The trained model set to evaluation mode.
    config : Configuration
        The configuration object used for training and exporting the model.

    Notes:
    ------
    - The function prints dataset statistics and model file names for verification.
    - Legend entries are dynamically created based on the provided `labels` list.
    - If `augment_data` is True, data augmentation is applied during training using `createTransforms(True)`.
    - The function validates dataset consistency before training.
    - After training, the model is saved and exported to ONNX, and metadata is written.
    """
    if maskdata is None:
        maskdata = [0.2, 0.3, 0.5]
    config = Configuration()
    print("Device: " + str(config.device))
    config.setSaveInterval(save_interval)
    config.setSavePath(save_path)
    config.setIsCrowd(False)
    config.setDatasetPaths(trainPath=trainDirectory, testPath=testDirectory)
    config.setFilePrefix("")
    config.setModelName(modelName)
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2 + 1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.setEpochs(epochs)
    config.setOnnxInfo(producer="Tygron", description=model_description)
    config.addLegendEntry("Background", 0, "#00000000")
    i = 1
    for label in labels:
        config.addLegendEntry(label, i, "#" + ''.join([random.choice('ABCDEF0123456789') for i in range(6)]))
        i += 1

    config.setOnnxMetaData(scoreThreshold=maskdata[0],
                           maskThreshold=maskdata[1],
                           strideFraction=maskdata[2])

    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    if augment_data:
        trainingDataset = ImageDataset(config, True, imageTransforms=createTransforms(True))
        testDataset = ImageDataset(config, False, createTransforms(False))
    else:
        trainingDataset = ImageDataset(config, True, createTransforms(False))
        testDataset = ImageDataset(config, False, createTransforms(False))

    print("Train Image count: " + str(trainingDataset.__len__()))
    print("Test Image count: " + str(testDataset.__len__()))

    if not trainingDataset.validateFiles():
        print("Inconsistent training dataset ")
        trainingDataset.validateFiles()

    if not testDataset.validateFiles():
        print("Inconsistent test dataset ")
        testDataset.validateFiles()

    print("Pytorch model name " + config.getPytorchModelFileName())
    print("Onnx file name " + config.getOnnxFileName())

    model = trainModel(config, trainingDataset, testDataset)
    model.eval()

    saveModel(config, model, epoch=epochs)

    exportOnnxModel(config, model)
    writeONNXMeta(config)

    return model, config

# create model template

In [5]:
import os
import torch
from pathlib import Path
import traceback

# Base directories
train_base_dir = Path("C:\\Users\\Tomkr\\Jaar2BlokD\\Tygron\\Tygron\\Datasets\\Train_data")  
test_base_dir = Path("C:\\Users\\Tomkr\\Jaar2BlokD\\Tygron\\Tygron\\Datasets\\Test_data")
models_save_dir = Path("C:\\Users\\Tomkr\\Jaar2BlokD\\Tygron\\Tygron\\Models\\")

# Ensure the models directory exists
os.makedirs(models_save_dir, exist_ok=True)

# Fixed parameters for all models
epochs = 1
labels = ["parking_space"]
augment = False
save_interval = 0

# List all subdirectories in the training dataset base directory
train_dataset_folders = [f for f in train_base_dir.iterdir() if f.is_dir()]
print(f"Found {len(train_dataset_folders)} training dataset folders")

# Process each dataset folder
for dataset_folder in train_dataset_folders:
    folder_name = dataset_folder.name
    print(f"\n{'='*50}\nProcessing dataset: {folder_name}\n{'='*50}")
    
    # Define paths for this dataset
    train_dir = dataset_folder  # The training folder itself
    
    # Look for corresponding test folder with the same name
    test_dir = test_base_dir / folder_name
    
    # If specific test folder doesn't exist, use the whole test directory
    if not test_dir.exists():
        print(f"No specific test folder for {folder_name}, using general test directory")
        test_dir = test_base_dir
    
    print(f"Using train directory: {train_dir}")
    print(f"Using test directory: {test_dir}")
    
    # Skip if training directory doesn't have content or test directory doesn't exist
    if not any(train_dir.iterdir()) or not test_dir.exists():
        print(f"Warning: Empty train directory or missing test directory for {folder_name}, skipping")
        continue
    
    # Create model name based on folder name
    model_name = f"{folder_name}_15.onnx"
    
    try:
        print(f"Creating model for {folder_name} with {epochs} epochs")
        
        # Clear CUDA cache if using GPU
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        model, config = createModel(
            str(train_dir),
            str(test_dir),
            model_name,
            epochs,
            labels,
            augment,
            str(models_save_dir),
            save_interval,
            maskdata=[0.2, 0.3, 0.5],
            model_description=f"Parking space detection model for {folder_name} dataset - 15 epochs"
        )
        print(f"✅ Successfully created and saved model for {folder_name}")
    except Exception as e:
        print(f"❌ Error creating model for {folder_name}: {str(e)}")
        print(f"Detailed error: {traceback.format_exc()}")
        # Continue with next dataset rather than stopping execution
        continue

print("\nAll datasets processed. Check the models directory for the created models.")

Found 6 training dataset folders

Processing dataset: AnnapaulownalaanCombo
No specific test folder for AnnapaulownalaanCombo, using general test directory
Using train directory: C:\Users\Tomkr\Jaar2BlokD\Tygron\Tygron\Datasets\Train_data\AnnapaulownalaanCombo
Using test directory: C:\Users\Tomkr\Jaar2BlokD\Tygron\Tygron\Datasets\Test_data
Creating model for AnnapaulownalaanCombo with 1 epochs
Device: cpu
Train Image count: 400
Test Image count: 1200
Pytorch model name C:\Users\Tomkr\Jaar2BlokD\Tygron\Tygron\ModelsAnnapaulownalaanCombo_15.onnx
Onnx file name C:\Users\Tomkr\Jaar2BlokD\Tygron\Tygron\ModelsAnnapaulownalaanCombo_15.onnx.onnx


c:\Users\Tomkr\Jaar2BlokD\Tygron\Tygron\Libraries\engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [  0/200]  eta: 0:45:39  lr: 0.000030  loss: 3.7909 (3.7909)  loss_classifier: 0.7295 (0.7295)  loss_box_reg: 0.0783 (0.0783)  loss_mask: 1.2809 (1.2809)  loss_objectness: 1.6026 (1.6026)  loss_rpn_box_reg: 0.0996 (0.0996)  time: 13.6994  data: 0.0560
Epoch: [0]  [ 10/200]  eta: 0:45:12  lr: 0.000281  loss: 3.5074 (3.3718)  loss_classifier: 0.6821 (0.6387)  loss_box_reg: 0.1188 (0.1548)  loss_mask: 0.9776 (1.0028)  loss_objectness: 1.3021 (1.3633)  loss_rpn_box_reg: 0.1680 (0.2122)  time: 14.2740  data: 0.0436
Epoch: [0]  [ 20/200]  eta: 0:38:38  lr: 0.000532  loss: 2.0668 (2.5867)  loss_classifier: 0.4477 (0.4953)  loss_box_reg: 0.1777 (0.1897)  loss_mask: 0.6982 (0.8265)  loss_objectness: 0.5104 (0.9239)  loss_rpn_box_reg: 0.1165 (0.1514)  time: 12.8371  data: 0.0436
Epoch: [0]  [ 30/200]  eta: 0:37:40  lr: 0.000783  loss: 1.5704 (2.1758)  loss_classifier: 0.3140 (0.4323)  loss_box_reg: 0.2240 (0.1961)  loss_mask: 0.5465 (0.7197)  loss_objectness: 0.2863 (0.6957)  loss_rp

KeyboardInterrupt: 